# Hands-On LangChain — Part 4: Vector Databases & Text Embeddings

The default path uses the pretrained all-MiniLM-L6-v2 model locally and a persistent Chroma database through current LangChain APIs. It needs a model download on the first run and no API key. Sample lecture passages are fixtures created for this notebook, not the original CS229 PDFs. The final opt-in section contains the complete updated OpenAI/CS229 workflow from the article.

We compare sentence vectors, split documents, retrieve passages, demonstrate duplicates, apply source filters, and reopen the saved database.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-chroma>=1.0,<2" "langchain-openai>=1.0,<2" "langchain-community>=0.4,<0.5" "langchain-text-splitters>=1.0,<2" pypdf numpy

## 2. Wrap the local pretrained embedding model

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
from langchain_core.embeddings import Embeddings
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

class LocalMiniLMEmbeddings(Embeddings):
    def __init__(self):
        self.model = ONNXMiniLM_L6_V2(preferred_providers=["CPUExecutionProvider"])
        self.model.DOWNLOAD_PATH = Path(tempfile.gettempdir()) / "tdb-minilm-model-cache"

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [vector.tolist() for vector in self.model(texts)]

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

embedding = LocalMiniLMEmbeddings()
print(f"Embedding dimensions: {len(embedding.embed_query('example'))}")

## 3. Compare sentence embeddings

In [ ]:
sentence1 = "My dog Rover likes to chase squirrels"
sentence2 = "Fluffy my cat, refuses to eat from a can"
sentence3 = "The chevy bolt accelerates to 60 mph in 6.7 seconds"
embedding1, embedding2, embedding3 = embedding.embed_documents([sentence1, sentence2, sentence3])
similarity_1_2 = float(np.dot(embedding1, embedding2))
similarity_1_3 = float(np.dot(embedding1, embedding3))
similarity_2_3 = float(np.dot(embedding2, embedding3))
print(f"Animal / animal similarity: {similarity_1_2:.4f}")
print(f"Animal / vehicle similarity: {similarity_1_3:.4f}")
print(f"Cat / vehicle similarity: {similarity_2_3:.4f}")

## 4. Create document fixtures with one deliberate duplicate

In [ ]:
from langchain_core.documents import Document
documents = [
    Document(page_content="Questions for the course TAs should be emailed to cs229-qa@cs.stanford.edu.", metadata={"source": "lecture-01", "page": 0}),
    Document(page_content="The course homework can be completed in MATLAB or Octave.", metadata={"source": "lecture-01", "page": 8}),
    Document(page_content="Supervised learning uses labeled examples for regression and classification.", metadata={"source": "lecture-03", "page": 1}),
    Document(page_content="The course homework can be completed in MATLAB or Octave.", metadata={"source": "lecture-01", "page": 8}),
]
print(f"Documents before deduplication: {len(documents)}")

## 5. Split documents, persist vectors, and retrieve passages

In [ ]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
splits = splitter.split_documents(documents)
persist_directory = tempfile.mkdtemp(prefix="tdb-vector-tutorial-")
vector_store = Chroma.from_documents(
    splits, embedding=embedding, collection_name="langchain-tutorial",
    persist_directory=persist_directory,
)
queries = [
    "What is the email address for the course TAs?",
    "What programming language is used for homework?",
    "What was said about supervised learning?",
]
for query in queries:
    result, distance = vector_store.similarity_search_with_score(query, k=1)[0]
    print(f"Query: {query}")
    print(f"Match: {result.page_content}")
    print(f"Metadata: {result.metadata}")
    print(f"Distance (lower is closer): {distance:.4f}")
    print()

## 6. Detect and remove duplicate-document failure cases

In [ ]:
matlab_text = "The course homework can be completed in MATLAB or Octave."
duplicate_results = vector_store.similarity_search("MATLAB or Octave homework", k=4)
duplicate_count = sum(result.page_content == matlab_text for result in duplicate_results)
unique_documents = list({
    (document.page_content, tuple(sorted(document.metadata.items()))): document
    for document in documents
}.values())
clean_store = Chroma.from_documents(
    unique_documents, embedding=embedding, collection_name="langchain-deduplicated",
    persist_directory=persist_directory,
)
clean_results = clean_store.similarity_search("MATLAB or Octave homework", k=3)
clean_duplicate_count = sum(result.page_content == matlab_text for result in clean_results)
print(f"Duplicate MATLAB results before cleanup: {duplicate_count}")
print(f"Documents after deduplication: {len(unique_documents)}")
print(f"Duplicate MATLAB results after cleanup: {clean_duplicate_count}")

## 7. Apply source constraints and reopen the persistent database

In [ ]:
filtered_results = vector_store.similarity_search(
    queries[2], k=3, filter={"source": "lecture-03"},
)
reopened_store = Chroma(
    collection_name="langchain-tutorial", embedding_function=embedding,
    persist_directory=persist_directory,
)
print(f"Filtered results from lecture-03: {len(filtered_results)}")
print(f"Documents after reopening Chroma: {len(reopened_store.get()['ids'])}")

## 8. Verify every result

In [ ]:
assert len(embedding1) == 384
assert similarity_1_2 > similarity_1_3
assert similarity_1_2 > similarity_2_3
assert len(vector_store.get()["ids"]) == 4
assert "cs229-qa@cs.stanford.edu" in vector_store.similarity_search(queries[0], k=1)[0].page_content
assert "MATLAB or Octave" in vector_store.similarity_search(queries[1], k=1)[0].page_content
assert "Supervised learning" in vector_store.similarity_search(queries[2], k=1)[0].page_content
assert duplicate_count == 2 and len(unique_documents) == 3 and clean_duplicate_count == 1
assert filtered_results and all(doc.metadata["source"] == "lecture-03" for doc in filtered_results)
assert len(reopened_store.get()["ids"]) == 4
assert "cs229-qa@cs.stanford.edu" in reopened_store.similarity_search(queries[0], k=1)[0].page_content
print("Embedding dimensions verified: 384")
print("Embedding relationships verified: 3")
print("Similarity-search queries verified: 3")
print("Duplicate cleanup verified: 4 documents -> 3 unique documents")
print("Source filter verified: lecture-03 only")
print("Chroma persistence verified: 4 documents after reopening")
print("All vector database and text embedding assertions passed.")

## 9. Optional original OpenAI and CS229 PDF workflow

This section includes every original tutorial code step with current imports. Set `RUN_HOSTED_EXAMPLE = True` only after placing the three lecture PDFs in `docs/cs229_lectures/` and configuring `OPENAI_API_KEY` in your environment. It makes paid embedding calls when enabled. The hosted path was checked for valid Python and current integration imports; live OpenAI results were not execution-tested. Historical article scores and counts are not assertions for this path. PDF splitting and Chroma persistence use the same APIs exercised by the local path.

The database directory is created afresh without removing your files. Chroma writes data automatically; there is no extra `persist()` call. Save `persist_directory` if you want to reopen this hosted index later.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    import numpy as np
    from dotenv import load_dotenv

    load_dotenv()
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY in your environment before running the hosted example.")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_openai import OpenAIEmbeddings
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    sentence1 = "My dog Rover likes to chase squirrels"
    sentence2 = "Fluffy my cat, refuses to eat from a can"
    sentence3 = "The chevy bolt accelerates to 60 mph in 6.7 seconds"

In [ ]:
if RUN_HOSTED_EXAMPLE:
    embedding1 = embedding.embed_query(sentence1)
    embedding2 = embedding.embed_query(sentence2)
    embedding3 = embedding.embed_query(sentence3)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print('The similarity between sentences one and two is' , np.dot(embedding1, embedding2))
    print('The similarity between sentences one and three is' , np.dot(embedding1, embedding3))
    print('The similarity between sentences two and three' , np.dot(embedding2, embedding3))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_community.document_loaders import PyPDFLoader

    # Load PDF
    loaders = [
        # Duplicate documents on purpose - messy data
        PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf"),
        PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture01.pdf"),
        PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture02.pdf"),
        PyPDFLoader("docs/cs229_lectures/MachineLearning-Lecture03.pdf")
    ]
    docs = []
    for loader in loaders:
        docs.extend(loader.load())

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # Split
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1500,
        chunk_overlap = 150
    )

    splits = text_splitter.split_documents(docs)
    print(len(splits))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_chroma import Chroma
    from tempfile import mkdtemp

    # A fresh directory per run avoids stale collections without deleting files.
    persist_directory = mkdtemp(prefix="langchain-cs229-chroma-")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    vectordb = Chroma.from_documents(
        documents=splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

    print(len(vectordb.get()["ids"]))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "What are the names and emails of the course TA?"
    docs = vectordb.similarity_search(question,k=3)
    print(len(docs))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(docs[0].page_content)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "what did they say about matlab?"
    docs = vectordb.similarity_search(question,k=5)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(docs[0])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(docs[1])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    question = "what did they say about supervised machine learning in the third lecture?"
    docs = vectordb.similarity_search(question,k=5)
    for doc in docs:
        print(doc.metadata)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(docs[4].page_content)